# Garmin Connect — data fetch & format

Connect to Garmin Connect, fetch each dataset, and return formatted results.

Set `GARMIN_EMAIL` and `GARMIN_PASSWORD` in the repo-root `.env` file.

## Setup

In [ ]:
import os
import sys
import time
from datetime import date, timedelta
from pathlib import Path
from typing import Any

NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT = NOTEBOOK_DIR.parents[1] if NOTEBOOK_DIR.name == "garmin-connect" else NOTEBOOK_DIR

env_path = REPO_ROOT / ".env"
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip())

try:
    from garminconnect import Garmin
except ImportError:
    import subprocess

    subprocess.check_call([sys.executable, "-m", "pip", "install", "garminconnect"])
    from garminconnect import Garmin

GARMIN_EMAIL = os.environ.get("GARMIN_EMAIL")
GARMIN_PASSWORD = os.environ.get("GARMIN_PASSWORD")

## Shared helpers

In [ ]:
def normalize_hr_zones(raw: Any) -> dict[int, float]:
    zones: dict[int, float] = {}
    if not isinstance(raw, list):
        return zones
    for entry in raw:
        if not isinstance(entry, dict) or "zoneNumber" not in entry:
            continue
        zones[int(entry["zoneNumber"])] = float(entry.get("secsInZone") or 0)
    return zones


def zones_to_minute_columns(zones: dict[int, float]) -> list[float | None]:
    if not zones:
        return [None] * 5
    return [round(zones.get(zone, 0) / 60, 2) for zone in range(1, 6)]


def running_activities(client: Garmin, start_date: str, end_date: str) -> list[dict[str, Any]]:
    activities = client.get_activities_by_date(start_date, end_date)
    return [
        a for a in activities
        if a.get("activityType", {}).get("typeKey") == "running"
    ]


def fetch_hr_zones(client: Garmin, activity_id: Any) -> dict[int, float]:
    return normalize_hr_zones(client.get_activity_hr_in_timezones(str(activity_id)))


def _primary_device_entry(device_map: dict[str, Any]) -> dict[str, Any]:
    if not isinstance(device_map, dict):
        return {}
    for entry in device_map.values():
        if isinstance(entry, dict) and entry.get("primaryTrainingDevice"):
            return entry
    for entry in device_map.values():
        if isinstance(entry, dict):
            return entry
    return {}


def parse_training_status(raw: dict[str, Any]) -> dict[str, Any]:
    status = _primary_device_entry(
        (raw.get("mostRecentTrainingStatus") or {}).get("latestTrainingStatusData") or {}
    )
    load = status.get("acuteTrainingLoadDTO") or {}
    balance = _primary_device_entry(
        (raw.get("mostRecentTrainingLoadBalance") or {}).get("metricsTrainingLoadBalanceDTOMap") or {}
    )
    return {
        "calendar_date": status.get("calendarDate"),
        "training_status": status.get("trainingStatusFeedbackPhrase"),
        "fitness_trend": status.get("fitnessTrend"),
        "acute_load": load.get("dailyTrainingLoadAcute"),
        "chronic_load": load.get("dailyTrainingLoadChronic"),
        "acwr": load.get("dailyAcuteChronicWorkloadRatio"),
        "acwr_status": load.get("acwrStatus"),
        "chronic_load_min": load.get("minTrainingLoadChronic"),
        "chronic_load_max": load.get("maxTrainingLoadChronic"),
        "load_balance_feedback": balance.get("trainingBalanceFeedbackPhrase"),
        "monthly_aerobic_low": round(balance.get("monthlyLoadAerobicLow", 0) or 0, 1),
        "monthly_aerobic_high": round(balance.get("monthlyLoadAerobicHigh", 0) or 0, 1),
        "monthly_anaerobic": round(balance.get("monthlyLoadAnaerobic", 0) or 0, 1),
    }

## Connect

In [ ]:
if not GARMIN_EMAIL or not GARMIN_PASSWORD:
    raise RuntimeError("Set GARMIN_EMAIL and GARMIN_PASSWORD in the repo-root .env file.")

client = Garmin(email=GARMIN_EMAIL, password=GARMIN_PASSWORD)
client.login()

## Profile

In [ ]:
def format_profile(client: Garmin) -> dict[str, Any]:
    profile = (client.get_user_profile() or {}).get("userData", {})
    training = parse_training_status(client.get_training_status(date.today().isoformat()))
    return {
        "weight": round(profile.get("weight", 0) / 1000, 2),
        "height": profile.get("height"),
        "birthDate": profile.get("birthDate"),
        "vo2MaxRunning": profile.get("vo2MaxRunning"),
        "lactateThresholdSpeed": round(
            (profile.get("lactateThresholdSpeed", 0) * 1000 / 60), 2
        ),
        "lactateThresholdHeartRate": profile.get("lactateThresholdHeartRate"),
        "availableTrainingDays": profile.get("availableTrainingDays", []),
        "preferredLongTrainingDays": profile.get("preferredLongTrainingDays", []),
        "training_status": training.get("training_status"),
        "acute_load": training.get("acute_load"),
        "chronic_load": training.get("chronic_load"),
        "acwr": training.get("acwr"),
        "acwr_status": training.get("acwr_status"),
    }

In [ ]:
profile = format_profile(client)
profile

## Activities

In [ ]:
ACTIVITY_HEADERS = [
    "name", "type", "distance_km", "duration_min", "avg_pace_min_per_km",
    "max_pace_min_per_km", "max_hr", "avg_hr", "aerobic_te", "anaerobic_te",
    "training_effect", "zone_1_min", "zone_2_min", "zone_3_min", "zone_4_min", "zone_5_min",
]


def format_activity_row(activity: dict[str, Any], zones: dict[int, float]) -> list[Any]:
    avg_speed = activity.get("averageSpeed")
    max_speed = activity.get("maxSpeed")
    return [
        activity.get("activityName"),
        activity.get("activityType", {}).get("typeKey"),
        round((activity.get("distance", 0) or 0) / 1000, 2),
        round((activity.get("movingDuration", 0) or 0) / 60, 2),
        round((1000 / avg_speed) / 60, 2) if avg_speed else None,
        round((1000 / max_speed) / 60, 2) if max_speed else None,
        activity.get("maxHR"),
        activity.get("averageHR"),
        round(activity.get("aerobicTrainingEffect", 0) or 0, 2),
        round(activity.get("anaerobicTrainingEffect", 0) or 0, 2),
        activity.get("trainingEffectLabel"),
        *zones_to_minute_columns(zones),
    ]


def format_activities(client: Garmin, days_back: int = 7) -> dict[str, Any]:
    today = date.today()
    activities = running_activities(
        client,
        (today - timedelta(days=days_back)).isoformat(),
        today.isoformat(),
    )
    rows = []
    for activity in activities:
        zones = fetch_hr_zones(client, activity.get("activityId"))
        rows.append(format_activity_row(activity, zones))
    return {"Garmin Activities": {"Headers": ACTIVITY_HEADERS, "Rows": rows}}

In [ ]:
activities = format_activities(client, days_back=7)
activities

## Race predictions

In [ ]:
def format_race_predictions(client: Garmin) -> dict[str, Any]:
    raw = client.get_race_predictions()
    return {
        "Garmin Race Predictions": {
            "Headers": ["5k_min", "10k_min", "half_marathon_h", "marathon_h"],
            "Rows": [[
                round((raw.get("time5K", 0) or 0) / 60, 2),
                round((raw.get("time10K", 0) or 0) / 60, 2),
                round((raw.get("timeHalfMarathon", 0) or 0) / 3600, 2),
                round((raw.get("timeMarathon", 0) or 0) / 3600, 2),
            ]],
        }
    }

In [ ]:
race_predictions = format_race_predictions(client)
race_predictions

## Events

In [ ]:
def format_events(client: Garmin, months_ahead: int = 12) -> dict[str, Any]:
    today = date.today()
    start_month = today.replace(day=1)
    event_rows = []
    seen_titles: set[str] = set()

    for offset in range(months_ahead):
        year = start_month.year + (start_month.month - 1 + offset) // 12
        month = (start_month.month - 1 + offset) % 12 + 1
        calendar = client.get_scheduled_workouts(year=year, month=month)
        if not isinstance(calendar, dict):
            continue
        for event in calendar.get("calendarItems", []):
            if not isinstance(event, dict) or event.get("itemType") != "event":
                continue
            event_date_str = event.get("date")
            if not event_date_str:
                continue
            try:
                event_date = date.fromisoformat(event_date_str)
            except ValueError:
                continue
            if event_date <= today:
                continue
            title = (event.get("title") or "").strip().lower()
            if title in seen_titles:
                continue
            seen_titles.add(title)
            target = event.get("completionTarget") or {}
            event_rows.append([
                event.get("title"), event.get("itemType"), event_date_str,
                target.get("value"), target.get("unit"), target.get("unitType"),
            ])

    return {
        "Garmin Events": {
            "Headers": ["title", "item_type", "date", "target_value", "target_unit", "target_unit_type"],
            "Rows": event_rows,
        }
    }

In [ ]:
events = format_events(client, months_ahead=12)
events

## Weekly stats

In [ ]:
WEEKLY_STATS_HEADERS = [
    "week_start", "week_end", "distance_km", "zone_1_min", "zone_2_min",
    "zone_3_min", "zone_4_min", "zone_5_min", "easy_min", "hard_min", "easy_pct",
]


def activity_date(activity: dict[str, Any]) -> date | None:
    start_time = activity.get("startTimeLocal", "")
    if not start_time:
        return None
    try:
        return date.fromisoformat(start_time[:10])
    except ValueError:
        return None


def easy_hard_from_zones(zones: dict[int, float]) -> tuple[float, float, float | None]:
    easy_secs = zones.get(1, 0) + zones.get(2, 0)
    hard_secs = zones.get(3, 0) + zones.get(4, 0) + zones.get(5, 0)
    total_secs = easy_secs + hard_secs
    easy_pct = round(100 * easy_secs / total_secs, 1) if total_secs else None
    return round(easy_secs / 60, 2), round(hard_secs / 60, 2), easy_pct


def weekly_stats_rows(
    activities: list[dict[str, Any]],
    activity_zones: dict[Any, dict[int, float]],
    today: date,
    num_blocks: int = 4,
) -> list[list[Any]]:
    rows: list[list[Any]] = []
    for block_index in range(num_blocks - 1, -1, -1):
        block_end = today - timedelta(days=block_index * 7)
        block_start = block_end - timedelta(days=6)
        block_distance_km = 0.0
        block_zones = {zone: 0.0 for zone in range(1, 6)}
        for activity in activities:
            act_date = activity_date(activity)
            if act_date is None or not (block_start <= act_date <= block_end):
                continue
            block_distance_km += (activity.get("distance", 0) or 0) / 1000
            zones = activity_zones.get(activity.get("activityId"), {})
            for zone, secs in zones.items():
                if 1 <= zone <= 5:
                    block_zones[zone] += secs
        easy_min, hard_min, easy_pct = easy_hard_from_zones(block_zones)
        rows.append([
            block_start.isoformat(), block_end.isoformat(), round(block_distance_km, 2),
            *[round(block_zones[zone] / 60, 2) for zone in range(1, 6)],
            easy_min, hard_min, easy_pct,
        ])
    return rows


def format_weekly_stats(client: Garmin, weeks: int = 4) -> dict[str, Any]:
    today = date.today()
    activities = running_activities(
        client,
        (today - timedelta(days=weeks * 7 - 1)).isoformat(),
        today.isoformat(),
    )
    activity_zones = {
        a["activityId"]: fetch_hr_zones(client, a["activityId"])
        for a in activities if a.get("activityId") is not None
    }
    return {
        "Garmin Weekly Stats": {
            "Headers": WEEKLY_STATS_HEADERS,
            "Rows": weekly_stats_rows(activities, activity_zones, today, num_blocks=weeks),
        }
    }

In [ ]:
weekly_stats = format_weekly_stats(client, weeks=4)
weekly_stats

## Training load — coach reference

Notes for system prompt rules. How to interpret `get_training_status` fields and decide whether training is building, maintaining, or backing off.

### Key fields

| Field | Garmin API name | Meaning |
|---|---|---|
| `acute_load` | `dailyTrainingLoadAcute` | Recent 7-day training strain (short-term fatigue) |
| `chronic_load` | `dailyTrainingLoadChronic` | Longer-term fitness base (~28-day rolling load) |
| `acwr` | `dailyAcuteChronicWorkloadRatio` | Acute ÷ chronic — the key ratio for load decisions |
| `training_status` | `trainingStatusFeedbackPhrase` | Garmin label, e.g. `PRODUCTIVE_3`, `MAINTAINING_4` |
| `acwr_status` | `acwrStatus` | Garmin's assessment, e.g. `OPTIMAL` |

Load decays each day without new training (Garmin uses exponential decay, similar to Banister/EWMA). Chronic load moves slowly; acute load reacts quickly to recent sessions.

### ACWR is a point-in-time snapshot

`acwr` from `get_training_status(today)` answers: **right now, how does recent strain compare to my fitness base?**

- **ACWR = 0.8** → recent load is 80% of chronic base → under-loading (recovery week, light week, or detraining)
- **ACWR ≈ 1.0** → recent load matches base → maintaining
- **ACWR 1.1–1.2** → acute above chronic → base should climb over coming weeks if sustained
- **ACWR > 1.3** → spike → injury risk; hold or reduce volume

Do **not** average ACWR across weeks — it blurs the signal. ACWR already embeds time decay; averaging it adds little value.

### What to track instead of average ACWR

| Metric | Use |
|---|---|
| **Current ACWR** | Immediate state: building, maintaining, or backing off |
| **Chronic load at week-end** | Is the fitness base actually rising over time? |
| **Week-over-week chronic delta %** | Did base load increase sustainably (~10% per week) or not? |

Km and HR zones alone do not show whether the body is absorbing the work. Chronic load trend + current ACWR together answer that.

### Progressive overload heuristic

Without new training, load decays daily. To **maintain** chronic load, keep feeding in similar weekly load. To **increase** chronic load (build fitness), acute must exceed chronic for a sustained period.

```
if ACWR < 1.0   → not building; chronic base flat or falling
if ACWR 1.0–1.3 → progressive overload zone (1.1–1.2 is the sweet spot)
if ACWR > 1.3   → spike; hold or reduce volume
```

A ~10% weekly increase in chronic load maps to keeping ACWR slightly above 1.0 without spiking past ~1.3. If km went up but chronic load is flat, intensity may have increased without the base absorbing it. If chronic is rising and ACWR stays ~1.1–1.2, training is building sustainably.

### Combining with weekly stats

- **Weekly stats** (km, HR zones) → what was done
- **Training load** (acute, chronic, ACWR) → what effect it had on the body

Use both when evaluating past weeks and planning increases.

## Training periodization — coach reference

Notes for system prompt rules. Volume progression, recovery cycles, event taper, and intensity split per week type.

### Are these rules accurate?

**Mostly yes**, with standard coaching caveats:

| Rule | Verdict | Notes |
|---|---|---|
| Max **+10% volume per build week** | ✅ Standard | Classic "10% rule" for mileage / weekly load. Cap increases even if the athlete feels good. |
| Every **4th week = recovery at 80%** | ✅ Standard | 3:1 or 4:1 build-to-recovery is widely used. 80% of the previous week is a common recovery target. |
| **2-week taper** before an event | ✅ Standard | 2–3 week tapers are normal. Your **−20% then −20%** model means each taper week is 80% of the prior week (64% of pre-taper peak after both steps). |
| **80/20 easy/hard** baseline | ✅ Standard | Polarized model: ~80% easy (Z1–Z2), ~20% hard (Z3+). Matches the easy/hard split already tracked in weekly HR zones. |

Caveats for the agent:
- These are **guidelines**, not laws — adjust for injury, illness, life stress, or ACWR > 1.3.
- **Taper overrides recovery** — if week 4 of a block is also taper week 1, taper wins.
- Taper cuts **volume**, not all intensity — keep short quality touches so legs stay sharp.
- The 10% cap applies to **build weeks only**, not recovery or taper weeks.

### Week-type decision (priority order)

```
1. Race week          (0 weeks to event)
2. Taper final        (1 week to event)   → volume × 0.64 of reference
3. Taper first        (2 weeks to event)  → volume × 0.80 of reference
4. Recovery           (every 4th training week, mod 4 == 0) → volume × 0.80 of previous week
5. Build              (everything else)   → volume up to +10% vs previous week
```

**Taper takes priority over recovery.** A 4th-week recovery is skipped when it falls inside the 2-week taper window.

### Formula

```python
from datetime import date
from math import floor

BASELINE_EASY_PCT = 80
BASELINE_HARD_PCT = 20


def weeks_until_event(today: date, event_date: date) -> int:
    """Full weeks from today until event (Monday-aligned)."""
    days = (event_date - today).days
    return max(0, days // 7)


def training_week_number(today: date, plan_start: date) -> int:
    """1-based week index since plan start (Monday-aligned)."""
    return floor((today - plan_start).days / 7) + 1


def week_plan(
    today: date,
    event_date: date | None,
    plan_start: date,
    previous_week_volume: float,
    reference_peak_volume: float,
) -> dict:
    """
    Returns week type, volume target, and intensity split for the agent.

    previous_week_volume  — actual km or load from last week
    reference_peak_volume — recent build-week peak, used for taper targets
    """
    week_num = training_week_number(today, plan_start)
    remaining = weeks_until_event(today, event_date) if event_date else None

    # --- taper / race (highest priority) ---
    if remaining is not None:
        if remaining == 0:
            return _plan("race", 0.30, 90, 10, previous_week_volume, reference_peak_volume)
        if remaining == 1:
            return _plan("taper_final", 0.64, 85, 15, previous_week_volume, reference_peak_volume)
        if remaining == 2:
            return _plan("taper_first", 0.80, 80, 20, previous_week_volume, reference_peak_volume)

    # --- recovery (every 4th week) ---
    if week_num % 4 == 0:
        return _plan("recovery", 0.80, 90, 10, previous_week_volume, reference_peak_volume)

    # --- build ---
    return _plan("build", 1.10, 80, 20, previous_week_volume, reference_peak_volume)


def _plan(week_type, multiplier, easy_pct, hard_pct, prev_vol, peak_vol):
    if week_type == "build":
        target = min(prev_vol * multiplier, prev_vol * 1.10)  # cap at +10%
    elif week_type in ("taper_first", "taper_final"):
        target = peak_vol * multiplier  # taper relative to recent peak
    else:
        target = prev_vol * multiplier  # recovery / race

    return {
        "week_type": week_type,
        "volume_target": round(target, 1),
        "volume_multiplier": multiplier,
        "easy_pct": easy_pct,
        "hard_pct": hard_pct,
    }
```

### Volume multipliers by week type

| Week type | Trigger | Volume vs reference | Max change |
|---|---|---|---|
| **build** | default | up to **+10%** vs previous week | `min(prev × 1.10, plan)` |
| **recovery** | `week_num % 4 == 0` | **80%** of previous week | −20% |
| **taper_first** | 2 weeks to event | **80%** of peak build week | −20% from peak |
| **taper_final** | 1 week to event | **64%** of peak (80% × 80%) | −36% from peak |
| **race** | event week | **~30%** — shakeout only | minimal |

Taper sequence example (peak week = 50 km):

| Weeks to event | Type | Target km |
|---|---|---|
| 3+ | build / recovery | normal progression |
| 2 | taper_first | 50 × 0.80 = **40 km** |
| 1 | taper_final | 50 × 0.64 = **32 km** |
| 0 | race | **~15 km** easy + race |

### Intensity split (easy % / hard %) per week type

Baseline polarized model: **80% easy / 20% hard** (Z1–Z2 vs Z3+).

| Week type | Easy % | Hard % | Rationale |
|---|---|---|---|
| **build** | 80 | 20 | Standard polarized distribution |
| **recovery** | 90 | 10 | Cut intensity first — drop intervals/tempo, keep easy miles |
| **taper_first** | 80 | 20 | Reduce volume but keep one short quality session |
| **taper_final** | 85 | 15 | Shorter, sharper efforts — maintain neuromuscular touch |
| **race** | 90 | 10 | Easy jogging only pre-race |

Agent guidance:
- On **recovery** weeks: prefer dropping hard sessions over cutting easy volume further.
- On **taper** weeks: never add new intensity; shorten existing quality work (fewer reps, not higher pace).
- Check **ACWR** alongside these rules — if ACWR > 1.3, hold volume regardless of planned week type.

### Linking to Garmin data

| Data | Use in periodization |
|---|---|
| `weekly_stats.distance_km` | Compare actual vs `volume_target` |
| `weekly_stats.easy_pct` / `hard_pct` | Compare actual intensity split vs target |
| `acute_load`, `chronic_load`, `acwr` | Validate the body absorbed the planned load |
| `get_events` → next event date | Drive `weeks_until_event` for taper logic |

A build week that hits +10% km but pushes ACWR above 1.3 was too aggressive — scale back next week even if the plan says build.

## Training load (current)

From `get_training_status(date)`. Acute load = recent 7-day strain. Chronic load = longer-term fitness base (~28 days). ACWR = acute / chronic.

In [ ]:
def format_training_load(client: Garmin, cdate: str | None = None) -> dict[str, Any]:
    cdate = cdate or date.today().isoformat()
    return parse_training_status(client.get_training_status(cdate))

In [ ]:
training_load = format_training_load(client)
training_load

## Training load (weekly history)

Samples `get_training_status` at each week-end to track how load evolved, plus rolling averages the agent can use to judge safe increases.

In [ ]:
TRAINING_LOAD_HEADERS = [
    "week_start", "week_end", "acute_load", "chronic_load", "acwr",
    "training_status", "acwr_status",
]


def format_training_load_weeks(client: Garmin, weeks: int = 4) -> dict[str, Any]:
    today = date.today()
    rows: list[list[Any]] = []
    acute_values: list[float] = []
    chronic_values: list[float] = []

    for block_index in range(weeks - 1, -1, -1):
        block_end = today - timedelta(days=block_index * 7)
        block_start = block_end - timedelta(days=6)
        parsed = parse_training_status(
            client.get_training_status(block_end.isoformat())
        )
        time.sleep(0.2)

        acute = parsed.get("acute_load")
        chronic = parsed.get("chronic_load")
        if acute is not None:
            acute_values.append(float(acute))
        if chronic is not None:
            chronic_values.append(float(chronic))

        rows.append([
            block_start.isoformat(),
            block_end.isoformat(),
            acute,
            chronic,
            parsed.get("acwr"),
            parsed.get("training_status"),
            parsed.get("acwr_status"),
        ])

    return {
        "Garmin Training Load": {
            "Headers": TRAINING_LOAD_HEADERS,
            "Rows": rows,
            "Averages": {
                "avg_acute_load": round(sum(acute_values) / len(acute_values), 1) if acute_values else None,
                "avg_chronic_load": round(sum(chronic_values) / len(chronic_values), 1) if chronic_values else None,
            },
        }
    }

In [ ]:
training_load_weeks = format_training_load_weeks(client, weeks=4)
training_load_weeks